# 4 · Collect Particle Annotations from H5 Files
Reads per-experiment `.h5` particle files, extracts annotation columns from the embedded JSON, and merges everything into a single Parquet file.

**Bug fixed:** `FileIndex` is now monotonically increasing across all files (original code reset it to 0 at the start of each chunk).

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
H5_DIR     = "/path/to/h5_files/"   # <-- set this before running
OUT_PARQUET = "./metadata/parquet_files/df_ann.parquet"
CHUNK_SIZE  = 50   # files flushed to a temp Parquet at a time

In [2]:
import os
import json
import h5py
import numpy as np
import pandas as pd

## Constants

In [3]:
EXTRACT_COLUMNS = [
    'ReconstructionResolution', 'EMD_Resolution', 'MolecularWeight_kDa',
    'ImagePixelSize', 'ImageSize', 'ClassResolution', 'ClassECA',
    '2DAlignmentClassESS', 'DefocusAngle', 'DefocusU', 'DefocusV',
    'ProteinClass', 'Symmetry', 'ClassNumber',
    'AngleRot', 'AngleTilt', 'AnglePsi',
    'Subunits', 'MaxDiameter', 'DNA', 'RNA',
]

# Known per-file annotation overrides
H5_OVERRIDES = {
    '10407_particles.h5': {'EMD_Resolution': 2.7},
}

## 1 · Collect annotations

In [4]:
h5_list = sorted(f for f in os.listdir(H5_DIR) if f.endswith('.h5'))
print(f"Found {len(h5_list)} H5 files")

chunk_paths   = []
chunk_frames  = []
global_index  = 0   # monotonically increasing across ALL files (bug fix)

for i, h5_filename in enumerate(h5_list):
    h5_path = os.path.join(H5_DIR, h5_filename)
    try:
        with h5py.File(h5_path, 'r') as f:
            df_exp = pd.DataFrame.from_dict(json.loads(f['annotations'][()].decode()))
    except Exception as exc:
        print(f"  ERROR reading {h5_filename}: {exc} — skipping")
        continue

    # Apply known overrides
    for col, val in H5_OVERRIDES.get(h5_filename, {}).items():
        df_exp[col] = val

    df_exp_ = df_exp[EXTRACT_COLUMNS].fillna(df_exp.loc[df_exp.index[0], EXTRACT_COLUMNS])
    df_exp_.insert(0, 'FileIndex',         global_index)
    df_exp_.insert(1, 'FileName',          h5_filename)
    df_exp_.insert(2, 'FileParticleIndex', np.arange(len(df_exp_)))

    chunk_frames.append(df_exp_)
    global_index += 1
    print(f'  [{global_index}/{len(h5_list)}] {h5_filename}', end='\r')

    # Flush chunk
    if (i + 1) % CHUNK_SIZE == 0 or i == len(h5_list) - 1:
        chunk_path = OUT_PARQUET.replace('.parquet', f'_chunk{len(chunk_paths)}.parquet')
        pd.concat(chunk_frames, ignore_index=True).to_parquet(chunk_path)
        chunk_paths.append(chunk_path)
        chunk_frames = []
        print(f"\n  Chunk {len(chunk_paths)} written → {chunk_path}")

print(f"\nProcessed {global_index} files")

Found 252 H5 files
  [50/252] 10271_particles.h5icles.h5
  Chunk 1 written → ./metadata/parquet_files/df_ann_chunk0.parquet
  [100/252] 10454_particles.h5cles.h5
  Chunk 2 written → ./metadata/parquet_files/df_ann_chunk1.parquet
  [150/252] 10947_cryoppp_particles.h5
  Chunk 3 written → ./metadata/parquet_files/df_ann_chunk2.parquet
  [200/252] 11720_particles.h5icles.h5
  Chunk 4 written → ./metadata/parquet_files/df_ann_chunk3.parquet
  [250/252] Kastritis_MIPS_particles.h5cles.h5
  Chunk 5 written → ./metadata/parquet_files/df_ann_chunk4.parquet
  [252/252] Kastritis_psII_2_particles.h5
  Chunk 6 written → ./metadata/parquet_files/df_ann_chunk5.parquet

Processed 252 files


## 2 · Concatenate chunks and save

In [5]:
df_ann = pd.concat([pd.read_parquet(p) for p in chunk_paths], ignore_index=True)
df_ann.to_parquet(OUT_PARQUET)
print(f"Saved {len(df_ann):,} rows → {OUT_PARQUET}")

for p in chunk_paths:
    os.remove(p)
print(f"Removed {len(chunk_paths)} temporary chunk files")

Saved 37,623,123 rows → ./metadata/parquet_files/df_ann.parquet
Removed 6 temporary chunk files


## 3 · Inspect

In [6]:
df_ann.head()

,FileIndex,FileName,FileParticleIndex,ReconstructionResolution,EMD_Resolution,MolecularWeight_kDa,ImagePixelSize,ImageSize,ClassResolution,ClassECA,...,ProteinClass,Symmetry,ClassNumber,AngleRot,AngleTilt,AnglePsi,Subunits,MaxDiameter,DNA,RNA
0,0,10004_particles.h5,0,7.88,6.0,293.14,1.08,320.0,9.642510,1.03,...,Viral Systems,C3,15.0,-2.068104,160.163437,-126.810539,12.0,180.0,0.0,0.0
1,0,10004_particles.h5,1,7.88,6.0,293.14,1.08,320.0,9.955498,1.97,...,Viral Systems,C3,27.0,-11.452702,37.757759,176.175400,12.0,180.0,0.0,0.0
2,0,10004_particles.h5,2,7.88,6.0,293.14,1.08,320.0,12.869015,1.19,...,Viral Systems,C3,12.0,110.746292,42.655224,152.476593,12.0,180.0,0.0,0.0
3,0,10004_particles.h5,3,7.88,6.0,293.14,1.08,320.0,9.404671,1.73,...,Viral Systems,C3,14.0,-26.400024,29.811657,-160.755890,12.0,180.0,0.0,0.0
4,0,10004_particles.h5,4,7.88,6.0,293.14,1.08,320.0,12.845575,2.40,...,Viral Systems,C3,37.0,-67.225204,31.797848,-155.622604,12.0,180.0,0.0,0.0


In [7]:
df_ann.describe()

,FileIndex,FileParticleIndex,ReconstructionResolution,EMD_Resolution,MolecularWeight_kDa,ImagePixelSize,ImageSize,ClassResolution,ClassECA,2DAlignmentClassESS,...,DefocusU,DefocusV,ClassNumber,AngleRot,AngleTilt,AnglePsi,Subunits,MaxDiameter,DNA,RNA
count,3.762312e+07,3.762312e+07,3.368289e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,...,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07,3.762312e+07
mean,1.324090e+02,1.120645e+05,3.500193e+00,3.374452e+00,5.946845e+02,1.024068e+00,3.214496e+02,9.219459e+00,1.242097e+00,1.533074e+00,...,1.744462e+04,1.714938e+04,2.446316e+01,1.651019e+00,9.525755e+01,-3.576555e-01,1.986453e+01,1.781158e+02,9.897060e-02,1.596765e-01
std,7.277361e+01,8.616125e+04,1.210975e+00,1.121338e+00,1.552509e+03,2.559697e-01,7.494228e+01,8.257123e+00,4.313045e-01,1.149261e+00,...,7.222549e+03,7.177734e+03,1.437643e+01,1.043662e+02,3.879604e+01,1.042219e+02,7.410056e+01,8.041433e+01,2.986225e-01,3.663058e-01
min,0.000000e+00,0.000000e+00,1.920000e+00,1.510000e+00,2.067000e+01,4.339290e-01,1.280000e+02,3.045692e+00,1.000000e+00,1.000000e+00,...,-3.106142e+04,-3.723793e+05,0.000000e+00,-1.800000e+02,1.205400e-02,-1.800000e+02,0.000000e+00,6.000000e+01,0.000000e+00,0.000000e+00
25%,7.000000e+01,4.091600e+04,2.820000e+00,2.800000e+00,1.429000e+02,8.400000e-01,2.560000e+02,6.041584e+00,1.000000e+00,1.000002e+00,...,1.226321e+04,1.201390e+04,1.200000e+01,-8.691403e+01,7.104591e+01,-9.080320e+01,2.000000e+00,1.200000e+02,0.000000e+00,0.000000e+00
50%,1.330000e+02,9.250000e+04,3.130000e+00,3.200000e+00,2.518600e+02,1.050000e+00,3.000000e+02,6.686344e+00,1.010000e+00,1.018309e+00,...,1.677011e+04,1.648869e+04,2.400000e+01,4.491120e-01,9.342699e+01,-9.531120e-01,5.000000e+00,1.500000e+02,0.000000e+00,0.000000e+00
75%,1.970000e+02,1.679210e+05,3.720000e+00,3.660000e+00,5.306800e+02,1.150000e+00,3.840000e+02,9.832233e+00,1.260000e+00,1.586569e+00,...,2.183415e+04,2.148962e+04,3.700000e+01,9.313121e+01,1.220138e+02,9.026585e+01,1.200000e+01,2.200000e+02,0.000000e+00,0.000000e+00
max,2.510000e+02,4.507200e+05,1.140000e+01,2.368000e+01,2.000000e+05,4.291071e+00,4.480000e+02,7.689600e+02,3.097000e+01,4.928275e+01,...,4.340204e+05,1.048005e+05,4.900000e+01,1.800000e+02,1.799928e+02,1.800000e+02,8.000000e+02,1.310000e+03,1.000000e+00,1.000000e+00
